In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder

In [2]:
data = pd.read_csv("../datasets/real_estate_properties_final_testing.csv")
df = data.copy()

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   price          24 non-null     int64  
 1   date_sold      24 non-null     str    
 2   suburb         24 non-null     str    
 3   num_bath       24 non-null     int64  
 4   num_bed        24 non-null     int64  
 5   num_parking    24 non-null     int64  
 6   property_size  24 non-null     float64
 7   type           24 non-null     str    
 8   source_url     24 non-null     str    
dtypes: float64(1), int64(4), str(4)
memory usage: 4.1 KB


In [4]:
df.drop(columns = "source_url", inplace = True)

In [5]:
df["date_sold"] = pd.to_datetime(
    df["date_sold"],
    format = "mixed"
)

df["day"] = df["date_sold"].dt.day
df["month"] = df["date_sold"].dt.month
df["year"] = df["date_sold"].dt.year

df.drop(columns = "date_sold")

,price,suburb,num_bath,num_bed,num_parking,property_size,type,day,month,year
0,2450000,mosman,2,3,2,299.0,Townhouse,22,8,2025
1,753500,mosman,1,1,1,57.0,Unit,21,8,2025
2,11360000,mosman,3,4,4,660.0,House,16,8,2025
3,5700000,mosman,3,5,2,468.0,House,9,8,2025
4,9650000,mosman,3,5,2,904.0,House,6,6,2025
5,9150000,mosman,3,6,3,660.0,House,6,6,2025
6,7100000,mosman,2,5,2,524.8,House,28,5,2025
7,4400000,mosman,1,3,1,339.0,House,28,5,2025
8,820000,parramatta,2,3,1,114.0,Apartment,17,6,2026
9,565000,parramatta,1,2,1,108.0,Unit,11,6,2026


In [6]:
for col in df.select_dtypes(include = str).columns:
    print(df[col].unique())

<ArrowStringArray>
['mosman', 'parramatta', 'blacktown']
Length: 3, dtype: str
<ArrowStringArray>
['Townhouse', 'Unit', 'House', 'Apartment']
Length: 4, dtype: str


In [7]:
onehotencoder = OneHotEncoder(sparse_output = False)
encoded_cols = onehotencoder.fit_transform(df[df.select_dtypes(include = str).columns].copy())

In [8]:
df[onehotencoder.categories_[0]] = encoded_cols[:, 0:3]
df[onehotencoder.categories_[1]] = encoded_cols[:, 3:]

In [9]:
df.drop(columns = ["date_sold", "suburb", "type"], inplace = True)
df["Block of units"] = 0.0
df["Duplex/semi-detached"] = 0.0
df = df[[
    "price",
    "num_bath",
    "num_bed",
    "num_parking",
    "property_size",
    "day",
    "month",
    "year",
    "blacktown",
    "mosman",
    "parramatta",
    "Apartment",
    "Block of units",
    "Duplex/semi-detached",
    "House",
    "Townhouse",
    "Unit"
]]

In [10]:
df = df.astype(float)
df.to_csv(
    "../datasets/real_estate_properties_final_testing_cleaned.csv",
    index = False,
    mode = "w"
)